In [0]:
%sql
SELECT * FROM pfnadds_labsca_unitaccuracy.unit_acc_weekly_detail LIMIT 100

In [0]:
%sql
 SELECT * FROM pfnadds_labsca_unitaccuracy.unit_acc_weekly_detail

In [0]:
%sql
select * from uc_prod_pfna_mdip_01.pfnadds_labsca_unitaccuracy.unitaccuracydetail_flna

In [0]:
%sql
 select distinct (materialtype) from uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.material_types_dim


In [0]:
%sql
select * from uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fri_pag_material_lookup_dim limit 100

In [0]:
%sql
describe table uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.ibp_source_destination_ratios_six_levels

In [0]:
%sql
select * from uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.ibp_source_destination_ratios_six_levels limit 100

In [0]:
%sql
CREATE OR REPLACE VIEW
    uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.sourcing_v1
AS
 
SELECT
    -- Sourcing identifier
    -- No source mapping is currently available.
    CAST(NULL AS STRING)                         AS `loc_inven_yrwk_key`,
 
    -- Supplier details
    -- Source mappings are not currently available.
    CAST(NULL AS STRING)                         AS `Supplier Code`,
    CAST(NULL AS STRING)                         AS `Supplier Name`,
 
    -- Ultimate source division
    -- Source mapping is not currently available.
    CAST(NULL AS STRING)                         AS `Ultimate Source Division`,
 
    -- Ultimate source location
    CAST(src.producer_locationname AS STRING)    AS `Ultimate Source Location Name`,
    CAST(src.producer_locationid AS BIGINT)      AS `Ultimate Source Location ID`,
 
    -- Immediate source location
    CAST(src.lvl_1_sourcelocationid AS BIGINT)    AS `Immediate Source Location ID`,
    CAST(src.lvl_1_sourcelocationname AS STRING)  AS `Immediate Source Location Name`,
 
    -- Period attributes
    -- Source mappings are not currently available.
    CAST(NULL AS STRING)                         AS `prd`,
    CAST(NULL AS STRING)                         AS `wk`,
 
    -- Ultimate source unit details
    -- Source mappings are not currently available.
    CAST(NULL AS BIGINT)                         AS `Ultimate Source Unit ID`,
    CAST(NULL AS STRING)                         AS `Ultimate Source Unit Name`,
 
    -- Ultimate source pod details
    -- Source mappings are not currently available.
    CAST(NULL AS BIGINT)                         AS `Ultimate Source Pod ID`,
    CAST(NULL AS STRING)                         AS `Ultimate Source Pod Name`,
 
    -- Sourcing hierarchy
    CAST(src.nodecount AS BIGINT)                AS `Sourcing Path Length`
 
FROM
    uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.ibp_source_destination_ratios_six_levels AS src;

In [0]:
%sql
select * from     uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.location_v1 limit 100


In [0]:
%sql

CREATE OR REPLACE VIEW
    uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.location_v1
AS
 
SELECT
    -- Location identifier and description
    CAST(loc.LocationId AS BIGINT)          AS `Location ID`,
    CAST(loc.LocationName AS STRING)        AS `Location Name`,
 
    -- iCAM-specific location classification
    -- Source mapping is not currently available.
    CAST(NULL AS STRING)                    AS `iCAM Location Type Description`,
 
    -- Geographic coordinates
    CAST(loc.LongitudeQuantity AS DOUBLE)   AS `Longitude`,
    CAST(loc.LatitudeQuantity AS DOUBLE)    AS `Latitude`,
 
    -- Third-party logistics indicator
    CAST(loc.location3plflag AS STRING)     AS `Third Party Logistics Flag`,
 
    -- Location geography
    CAST(loc.CityName AS STRING)            AS `City Name`,
    CAST(loc.TerritoryCode AS STRING)       AS `State Code`,
    CAST(loc.PostalArea AS STRING)          AS `Zip Code`,
 
    -- Regional hierarchy
    CAST(loc.DivisionCode AS BIGINT)        AS `Region ID`,
    CAST(loc.DivisionName AS STRING)        AS `Region`
 
FROM
    uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim AS loc;







In [0]:
%sql

WITH source_calendar AS (
    SELECT
        CalendarDate,
        Year,
        Quarter,
        Period,
        Week,
        WeekInYearNumber,
        Year_Period,
        Year_PeriodXWeek_Padded,
        WeekStartDate
    FROM uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.time_day_dim
),

-- Get the current fiscal year, quarter, period and week.
current_context AS (
    SELECT
        Year AS current_fiscal_year,
        Quarter AS current_fiscal_quarter,
        Period AS current_fiscal_period,
        WeekStartDate AS current_week_start_date
    FROM source_calendar
    WHERE CalendarDate = CURRENT_DATE()
),

-- Retain one record per fiscal year-period.
calendar_base AS (
    SELECT
        sc.*,

        CAST(sc.Year_Period AS STRING) AS year_week_id,

        ROW_NUMBER() OVER (
            PARTITION BY sc.Year_Period
            ORDER BY sc.CalendarDate
        ) AS row_num

    FROM source_calendar sc
)

SELECT
    CAST(cb.Year AS BIGINT) AS `Year`,

    CAST(cb.Quarter AS BIGINT) AS `Quarter`,

    CAST(cb.Year_Period AS STRING) AS `Year Period`,

    CAST(NULL AS STRING) AS `Period Description`,

    CAST(cb.Period AS STRING) AS `Period`,

    CAST(cb.Week AS BIGINT) AS `Week In Period`,

    CAST(cb.year_week_id AS STRING) AS `Year Week`,

    CAST(cb.WeekInYearNumber AS BIGINT) AS `Week`,

    -- Week offset relative to the current fiscal week.
    CAST(
        DATEDIFF(
            cb.WeekStartDate,
            cc.current_week_start_date
        ) / 7
        AS BIGINT
    ) AS `Week Timing`,

    CAST(NULL AS STRING) AS `Time Frame`,

    CAST(NULL AS BIGINT) AS `inv_max_yr_wk_id`,

    CAST(NULL AS BIGINT) AS `projinv_max_yr_wk_id`,

    cb.Year_PeriodXWeek_Padded AS `Period Week Description`,

    -- Period offset based on 13 fiscal periods per year.
    CAST(
        ((cb.Year - cc.current_fiscal_year) * 13)
        + (cb.Period - cc.current_fiscal_period)
        AS BIGINT
    ) AS `Period Timing`,

    CAST(cb.WeekStartDate AS TIMESTAMP) AS `fscl_wk_start_dte`,

    -- Maximum Year Week within each fiscal period.
    MAX(cb.year_week_id) OVER (
        PARTITION BY cb.Year, cb.Period
    ) AS `max_yr_wk_id_in_prd`,

    CAST(NULL AS BIGINT) AS `io_max_yr_wk_id`,

    -- Fiscal year offset relative to the current year.
    CAST(
        cb.Year - cc.current_fiscal_year
        AS BIGINT
    ) AS `Year Timing`,

    -- Quarter offset based on four fiscal quarters per year.
    CAST(
        ((cb.Year - cc.current_fiscal_year) * 4)
        + (cb.Quarter - cc.current_fiscal_quarter)
        AS BIGINT
    ) AS `Quarter Timing`

FROM calendar_base cb

LEFT JOIN current_context cc
    ON 1 = 1

WHERE cb.row_num = 1

ORDER BY
    cb.Year,
    cb.Period;


In [0]:
%sql
WITH source_calendar AS (
    SELECT
        CalendarDate,
        Year,
        Quarter,
        Period,
        Week,
        WeekInYearNumber,
        Year_Period,
        Year_PeriodXWeek_Padded,
        WeekStartDate
    FROM uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.time_day_dim
),
 
-- Get the current fiscal year, quarter, period and week.
current_context AS (
    SELECT
        Year AS current_fiscal_year,
        Quarter AS current_fiscal_quarter,
        Period AS current_fiscal_period,
        WeekStartDate AS current_week_start_date
    FROM source_calendar
    WHERE CalendarDate = CURRENT_DATE()
),
 
-- Generate the fiscal year-week key and select one row per week.
calendar_base AS (
    SELECT
        sc.*,
 
        CAST(
            (sc.Year * 100) + sc.WeekInYearNumber
            AS BIGINT
        ) AS year_week_id,
 
        ROW_NUMBER() OVER (
            PARTITION BY sc.Year, sc.WeekInYearNumber
            ORDER BY sc.CalendarDate
        ) AS row_num
 
    FROM source_calendar sc
),
 
-- Get the last fiscal week within each year-period.
calendar_enriched AS (
    SELECT
        cb.*,
 
        MAX(cb.year_week_id) OVER (
            PARTITION BY cb.Year, cb.Period
        ) AS max_year_week_in_period
 
    FROM calendar_base cb
)
 
SELECT
    CAST(cb.Year AS BIGINT) AS `Year`,
 
    CAST(cb.Quarter AS BIGINT) AS `Quarter`,
 
    CAST(cb.Year_Period AS STRING) AS `Year Period`,
 
    CAST(NULL AS STRING) AS `Period Description`,
 
    CAST(cb.Period AS STRING) AS `Period`,
 
    CAST(cb.Week AS BIGINT) AS `Week In Period`,
 
    cb.year_week_id AS `Year Week`,
 
    CAST(cb.WeekInYearNumber AS BIGINT) AS `Week`,
 
    -- Week offset relative to the current fiscal week.
    CAST(
        DATEDIFF(
            cb.WeekStartDate,
            cc.current_week_start_date
        ) / 7
        AS BIGINT
    ) AS `Week Timing`,
 
    CAST(NULL AS STRING) AS `Time Frame`,
 
    CAST(NULL AS BIGINT) AS `inv_max_yr_wk_id`,
 
    CAST(NULL AS BIGINT) AS `projinv_max_yr_wk_id`,
 
    cb.Year_PeriodXWeek_Padded AS `Period Week Description`,
 
    -- Period offset based on 13 fiscal periods per year.
    CAST(
        ((cb.Year - cc.current_fiscal_year) * 13)
        + (cb.Period - cc.current_fiscal_period)
        AS BIGINT
    ) AS `Period Timing`,
 
    CAST(cb.WeekStartDate AS TIMESTAMP) AS `fscl_wk_start_dte`,
 
    cb.max_year_week_in_period AS `max_yr_wk_id_in_prd`,
 
    CAST(NULL AS BIGINT) AS `io_max_yr_wk_id`,
 
    -- Fiscal year offset relative to the current year.
    CAST(
        cb.Year - cc.current_fiscal_year
        AS BIGINT
    ) AS `Year Timing`,
 
    -- Quarter offset based on four fiscal quarters per year.
    CAST(
        ((cb.Year - cc.current_fiscal_year) * 4)
        + (cb.Quarter - cc.current_fiscal_quarter)
        AS BIGINT
    ) AS `Quarter Timing`
 
FROM calendar_enriched cb
 
LEFT JOIN current_context cc
    ON 1 = 1
 
WHERE cb.row_num = 1
 
ORDER BY
    cb.Year,
    cb.WeekInYearNumber;

In [0]:
%sql
--select * from uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.material_dim where MaterialId = 300039893
Select material_id,count_description,brand_description,sub_brand_description,product_line_description,flavor_description from uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fri_pag_material_lookup_dim where material_id='1600000194';

In [0]:
%sql
--describe uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.material_dim
--select 

--distinct(MaterialTypeCode), 
--MaterialTypeName
--distinct(unit)
 --from 
--uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.material_dim 
--describe table uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fri_pag_material_lookup_dim
--select distinct(material_type) from  uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.material
--select * from uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fri_pag_material_lookup_dim
--where material_id = '300039893'

Select materialid,brandname,subbrandname,productlinename,flavorname from uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.material_dim where MaterialId=1600000194
--SHOW create table uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fri_pag_material_lookup_dim 

--describe table uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.material_dim


In [0]:
%sql
--describe uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.material_types_dim
--select 
--distinct(materialtypename)
--distinct(materialid), 
--distinct(materialtypecode)
--materialtypename,
--distinct(innovationtype)
--materialsubtypecode,
--materialsubtypename,
--materialname

--from
--uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.material_types_dim
--limit 100

--select * from uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.material_types_dim
select * from uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.material_types_dim
where materialtype = 'CLUB'



In [0]:
%sql
--describe uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim

select distinct (LocationTypeCode),LocationTypeName from
uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim
limit 100


In [0]:
%sql
--select Distinct(LocationTypeName) from
describe table uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim 
--limit 100

In [0]:
%sql
select Distinct(SourceLocationTypeCode),SourceLocationTypeName from
uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim
limit 100


In [0]:
%sql
select * from
uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.sca_location_dim
limit 100

In [0]:
%sql
--select * 
--from 
 show create table uc_prod_pfna_mdip_01.pfna_certified_masterdata_internal.time_day_dim 
--where 
--Year = 2026
--limit 100

In [0]:
%sql
select * from uc_prod_pfna_mdip_01.pfnadds_labsca_commondim.ibp_source_destination_ratios_six_levels 
where 
materialid = '300020235' 
and 
year_periodxweek = '2026_1X4' 
and 
nodecount = 4
and
lvl_1_sourcelocationid is not null 
 and
lvl_2_sourcelocationid is not null



In [0]:
%sql
select distinct demandaggregatecategory, demandaggregatedescription from  uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fillratedetail_flna
where SOURCE_SYSTEM = 'PGT'
limit 1000

In [0]:
%sql
select * from  uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.pgt_cut_reason_codes

-- in this whatever has null value for compliance impacting flag are not counted for fill rates


In [0]:
%sql
select * from  uc_prod_pfna_mdip_01.pfna_certified_supplychain_internal.fillratedetail_flna
where SOURCE_SYSTEM = 'PGT'
limit 1000

--raw fill rate =shippedamount /allorderedamountv --- not much used used mostly at costco
--orders may be cut as a part of order cleansing process -- RSR? may order discontinued items etc
-- Fill rate (powerBI) = shippedamount/complianceorderedamount
-- compliance ordered amount = shippedamount + compliancecutamount (maybe if 80% of a case is ordered 100% may be shipped)